### [한번 해보기] 음식 리뷰 데이터 활용 유사도 검색
- corpus -> embedding vector -> 유사도 기반 검색

In [ ]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import os
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
client = OpenAI()

ffr_input = pd.read_csv('./data/fine_food_reviews_1k.csv')
ffr_input

,Unnamed: 0,Time,ProductId,UserId,Score,Summary,Text
0,0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...
...,...,...,...,...,...,...,...
995,995,1351209600,B004OQLIHK,AKHQMSUORSA91,5,Delicious!,I have ordered these raisins multiple times. ...
996,996,1351209600,B0006349W6,A21BT40VZCCYT4,5,Good Training Treat,My dog will come in from outside when I am tra...
997,997,1351209600,B00611F084,A6D4ND3C3BCYV,5,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...
998,998,1351209600,B005QKH5HA,A3LR9HCV3D96I3,5,Party Peanuts,Great product for the price. Mix with the Asia...


In [3]:
def text_to_embedding(texts, model='text-embedding-3-small'):
    texts = [text.replace('\n', ' ') for text in texts]
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [data.embedding for data in response.data]

In [16]:
# Summary (title), Text (content) ---> Content -> Embedding Vector (.csv 저장)
# Content -> Embedding Vector (.csv 저장)

summary_text_list = []
for i in range(len(ffr_input)):
    summary_text = f'{ffr_input["Summary"][i]} : {ffr_input["Text"][i]}'
    summary_text_list.append(summary_text)
embedding_list = text_to_embedding(summary_text_list)
# 사용자 입력 (best coffee) -> Embedding Vector2
input_text = 'best coffee'
# input_text = input('검색어를 입력하세요: ')
input_embedding = text_to_embedding([input_text])[0]
# 유사도 계산 -> 유사도가 높은 것
all_embedding = np.array(embedding_list)
similarities = cosine_similarity(all_embedding, [input_embedding])
ffr_input['similarity'] = similarities.flatten()
ffr_input.sort_values(by='similarity', ascending=False, inplace=True)
ffr_input.head(1)

,Unnamed: 0,Time,ProductId,UserId,Score,Summary,Text,similarity
923,923,1351209600,B007TJGZ54,A29BJSTYH9W3JI,5,super coffee,Great coffee and so easy to brew. This coffee...,0.612428
